In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load data

In [3]:
from datasets import load_dataset

ds = load_dataset("HuggingFaceM4/ChartQA")

C:\Users\Sonic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds["train"].column_names

['image', 'query', 'label', 'human_or_machine']

In [5]:
train_ds = ds["train"]
val_ds   = ds["val"]
test_ds  = ds["test"]

train_ds[0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=422x359>,
 'query': 'Is the value of Favorable 38 in 2015?',
 'label': ['Yes'],
 'human_or_machine': 0}

## Gemini

In [6]:
import os
import pandas as pd
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [16]:
# Model and sampling settings
MODEL = "gemini-2.5-flash"
N = 100                                  # number of samples

# Output settings
OUTPUT_FILE = "data/results.csv"
os.makedirs("data", exist_ok=True)

# Gemini Rate Settings
SAVE_EVERY = 5                          # checkpoint frequency
SLEEP_SECONDS = 2                      

In [17]:
import os
import time

# ---------------------------
# Load existing results (if any)
# ---------------------------
if os.path.exists(OUTPUT_FILE):
    df_existing = pd.read_csv(OUTPUT_FILE)
    processed_indices = set(df_existing["index"].tolist())
    results = df_existing.to_dict("records")
    start_i = max(processed_indices) + 1
    print(f"Resuming from index {start_i} ({len(processed_indices)} already done)")
else:
    results = []
    processed_indices = set()
    start_i = 0
    print("Starting fresh run")

# ---------------------------
# Evaluation loop
# ---------------------------

for i in range(start_i, min(N, len(train_ds))):
    if i in processed_indices:
        continue

    print(f"Processing sample {i+1:6d}/{N}")

    example = train_ds[i]
    image = example["image"]
    question = example["query"]
    true_label = example["label"][0]
    question_type = example["human_or_machine"] # 0/"human" or 1/"machine"

    prompt = f"""Answer the question about this chart.
                Question: {question}
                Provide only the final answer.
            """

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=[prompt, image],
        )
        prediction = (response.text or "").strip()
    except Exception as e:
        print(f"Error at index {i+1}")
        prediction = "ERROR"

        # redo this iteration
        i -= 1
        continue

    correct = int(
        str(prediction).strip().lower() ==
        str(true_label).strip().lower()
    )

    results.append({
        "index": i,
        "query": question,
        "question_type": question_type,
        "true_label": true_label,
        "prediction": prediction,
        "correct": correct
    })

    # ---------------------------
    # Checkpoint save
    # ---------------------------
    if (len(results) % SAVE_EVERY) == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
        print(f"Checkpoint saved ({len(results)} rows)")

    time.sleep(SLEEP_SECONDS)

# ---------------------------
# Final save
# ---------------------------
pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
print(f"Finished. Saved {len(results)} rows to {OUTPUT_FILE}")

Resuming from index 5 (5 already done)
Processing sample      6/100
Processing sample      7/100
Processing sample      8/100
Processing sample      9/100
Processing sample     10/100
Checkpoint saved (10 rows)
Processing sample     11/100
Processing sample     12/100
Processing sample     13/100
Processing sample     14/100
Processing sample     15/100
Checkpoint saved (15 rows)
Processing sample     16/100
Processing sample     17/100
Processing sample     18/100
Processing sample     19/100
Processing sample     20/100
Checkpoint saved (20 rows)
Processing sample     21/100
Processing sample     22/100
Processing sample     23/100
Processing sample     24/100
Processing sample     25/100
Checkpoint saved (25 rows)
Processing sample     26/100
Processing sample     27/100
Processing sample     28/100
Processing sample     29/100
Processing sample     30/100
Checkpoint saved (30 rows)
Processing sample     31/100
Processing sample     32/100
Processing sample     33/100
Processing sam